In [1]:
import pandas as pd
import torch
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from transformers import Trainer
from transformers import TrainingArguments
from transformers import AutoModelForSequenceClassification

In [2]:
from datasets import load_dataset

# 1. Load the dataset
dataset = load_dataset("SetFit/sst5")

# 2. Define the mapping function
def map_to_custom_labels(example):
    original_label = example['label']
    if original_label in [3, 4]:
        example['custom_label'] = 0  # Positive
    elif original_label in [0, 1]:
        example['custom_label'] = 1  # Negative
    else:
        example['custom_label'] = 2  # Neutral
    return example

# 3. Apply the mapping and clean up columns
mapped_dataset = dataset.map(map_to_custom_labels)
mapped_dataset = mapped_dataset.remove_columns(["label", "label_text"]) # Removed "text_id" and added "label_text"
mapped_dataset = mapped_dataset.rename_column("custom_label", "label")

# 4. Shuffle and select exactly 5,000 examples from the training split
# Using a fixed seed (like 42) ensures you get the same 5,000 rows every time you run the script
small_train_dataset = mapped_dataset['train'].shuffle(seed=42).select(range(1500))

print(f"Training set size: {len(small_train_dataset)}")
print(small_train_dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/421 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: 0.00B [00:00, ?B/s]

dev.jsonl: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/2210 [00:00<?, ? examples/s]

Map:   0%|          | 0/8544 [00:00<?, ? examples/s]

Map:   0%|          | 0/1101 [00:00<?, ? examples/s]

Map:   0%|          | 0/2210 [00:00<?, ? examples/s]

Training set size: 1500
{'text': '-lrb- a -rrb- slummer .', 'label': 1}


In [3]:
small_train_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 1500
})

In [4]:
display(small_train_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 1500
})

In [5]:
import pandas as pd

# Convert the small_train_dataset to a pandas DataFrame to view its contents
small_train_df = pd.DataFrame(small_train_dataset)

# Display the first few rows of the DataFrame
display(small_train_df.head(10))

,text,label
0,-lrb- a -rrb- slummer .,1
1,"this charming , thought-provoking new york fes...",0
2,"the ring is worth a look , if you do n't deman...",2
3,vereté has a whip-smart sense of narrative blu...,0
4,"highly irritating at first , mr. koury 's pass...",2
5,"the pain , loneliness and insecurity of the sc...",0
6,it is supremely unfunny and unentertaining to ...,1
7,eisenstein lacks considerable brio for a film ...,1
8,"kitschy , flashy , overlong soap opera .",1
9,"there are some fairly unsettling scenes , but ...",1


In [6]:
# Prepare the data for the model from the new dataset
texts = list(small_train_df["text"])
labels = list(small_train_df["label"])
print(f"Total texts count: {len(texts)}")
print(f"Total labels count: {len(labels)}")

Total texts count: 1500
Total labels count: 1500


In [7]:
type(texts)

list

In [8]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)

In [9]:


# -----------------------------
# Load tokenizer
# -----------------------------

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:

# -----------------------------
# Dataset class
# -----------------------------

class ReviewDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)


train_dataset = ReviewDataset(train_encodings, train_labels)
test_dataset = ReviewDataset(test_encodings, test_labels)


In [11]:

# -----------------------------
# Train or load model
# -----------------------------

if not os.path.exists("sentiment_model"):

    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=3
    )

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=4
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset
    )

    trainer.train()

    trainer.save_model("sentiment_model")
    tokenizer.save_pretrained("sentiment_model")

else:

    model = AutoModelForSequenceClassification.from_pretrained("sentiment_model")

    trainer = Trainer(model=model)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Step,Training Loss
500,0.826115


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:

# -----------------------------
# Evaluate model
# -----------------------------

predictions = trainer.predict(test_dataset)

preds = predictions.predictions.argmax(axis=1)
true_labels = predictions.label_ids

precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    preds,
    average="weighted"
)

accuracy = accuracy_score(true_labels, preds)

loss = predictions.metrics["test_loss"]

print("\nModel Evaluation Metrics:")
print("Loss:", loss)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Model Evaluation Metrics:
Loss: 1.1199718713760376
Accuracy: 0.7066666666666667
Precision: 0.7047820387742338
Recall: 0.7066666666666667
F1 Score: 0.704548024470505

Enter movie review: This movie was an absolute banger.
Sentiment: positive


In [29]:
# -----------------------------
# Predict sentiment
# -----------------------------

sentiment_labels = ["positive", "negative", "neutral"]

model.eval()

review = input("\nEnter movie review: ")

inputs = tokenizer(
    review,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print("Sentiment:", sentiment_labels[prediction])


Enter movie review: meh
Sentiment: neutral
